# 03 Regression First Model (New Notebook)

## What changes first (before coding)

This notebook changes the ML task from binary late/not-late to **regression on delivery margin days**.

- Old task: predict `is_late` directly.
- New task: predict `delivery_margin_days = actual_delivery_days - estimated_delivery_days`.

Why this is better:
- It models the size of delay/earliness, not just a binary flag.
- It keeps more signal for ML than residual binary late labels.
- It still supports operations decisions.

## ML roles in this new setup

1. **Prediction role:** estimate expected margin days for each order.
2. **Risk role:** convert predicted margin to probability/risk bands and late flags.
3. **Decision role:** apply capacity-constrained thresholds (Top-K or cap) for intervention.

This keeps strong ML depth while staying business-ready.

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.isotonic import IsotonicRegression

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../datasets/processed/olist_merged_clean.csv")
df = pd.read_csv(DATA_PATH)

for c in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

print("Loaded shape:", df.shape)
print("Late rate (%):", round(df["is_late"].mean() * 100, 2))
print("XGBoost available:", HAS_XGB)

Loaded shape: (96189, 34)
Late rate (%): 7.94
XGBoost available: True


In [7]:
work = df.copy()

# Core regression targets
work["actual_delivery_days"] = (
    work["order_delivered_customer_date"] - work["order_purchase_timestamp"]
).dt.days

work["estimated_delivery_days"] = (
    work["order_estimated_delivery_date"] - work["order_purchase_timestamp"]
).dt.days

# Positive => delivered before estimate, Negative => delivered after estimate
work["delivery_margin_days"] = (
    work["order_estimated_delivery_date"] - work["order_delivered_customer_date"]
).dt.days

# Safety cleaning on target logic
work = work.replace([np.inf, -np.inf], np.nan)
work = work.dropna(subset=["order_purchase_timestamp", "actual_delivery_days", "delivery_margin_days"])
work = work[(work["actual_delivery_days"] >= 0) & (work["actual_delivery_days"] <= 120)]
work = work[(work["delivery_margin_days"] >= -120) & (work["delivery_margin_days"] <= 120)]

# Time-based split boundaries
train_end = work["order_purchase_timestamp"].quantile(0.70)
val_end = work["order_purchase_timestamp"].quantile(0.85)

split = np.where(
    work["order_purchase_timestamp"] <= train_end,
    "train",
    np.where(work["order_purchase_timestamp"] <= val_end, "val", "test"),
)
work["split"] = split

print("Prepared shape:", work.shape)
print(work[["actual_delivery_days", "delivery_margin_days"]].describe().round(2))
print("Split counts:\n", work["split"].value_counts())

Prepared shape: (96142, 36)
       actual_delivery_days  delivery_margin_days
count              96142.00              96142.00
mean                  12.01                 11.00
std                    8.98                  9.58
min                    0.00               -106.00
25%                    6.00                  6.00
50%                   10.00                 12.00
75%                   15.00                 16.00
max                  118.00                108.00
Split counts:
 split
train    67299
test     14422
val      14421
Name: count, dtype: int64


In [14]:
TARGET = "delivery_margin_days"

leakage_cols = {
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "actual_delivery_days",
    "delivery_margin_days",
    "delivery_delay_days",
    "is_late",
    "order_id",
}

# Exclude fields that are available only after the order outcome.
post_outcome_tokens = ("delivered", "review", "status")

candidate_cols = [
    c
    for c in work.columns
    if c not in leakage_cols
    and c not in {"split"}
    and not any(t in c.lower() for t in post_outcome_tokens)
]

X_train = work.loc[work["split"] == "train", candidate_cols].copy()
y_train = work.loc[work["split"] == "train", TARGET].copy()

X_val = work.loc[work["split"] == "val", candidate_cols].copy()
y_val = work.loc[work["split"] == "val", TARGET].copy()

X_test = work.loc[work["split"] == "test", candidate_cols].copy()
y_test = work.loc[work["split"] == "test", TARGET].copy()

for frame in (X_train, X_val, X_test):
    for c in frame.columns:
        if pd.api.types.is_datetime64_any_dtype(frame[c]):
            frame[c] = frame[c].astype("int64", copy=False) // 10**9

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler(with_mean=False)),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

print(f"Feature columns: {len(candidate_cols)} | Numeric: {len(num_cols)} | Categorical: {len(cat_cols)}")

Feature columns: 28 | Numeric: 22 | Categorical: 6


In [15]:
models = {
    "elastic_net": ElasticNet(alpha=0.0008, l1_ratio=0.2, random_state=42, max_iter=4000),
    "rf_reg": RandomForestRegressor(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
    ),
}

if HAS_XGB:
    models["xgb_reg"] = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )

val_rows = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)

    p_train = pipe.predict(X_train)
    p_val = pipe.predict(X_val)

    row = {
        "model": name,
        "train_mae": mean_absolute_error(y_train, p_train),
        "val_mae": mean_absolute_error(y_val, p_val),
        "val_rmse": np.sqrt(mean_squared_error(y_val, p_val)),
        "val_r2": r2_score(y_val, p_val),
    }
    val_rows.append(row)
    fitted[name] = pipe

val_table = pd.DataFrame(val_rows).sort_values("val_mae")
val_table.round(4)

C:\Users\Devyansh\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 76005.89023778867, tolerance: 627.8608478075453
  model = cd_fast.sparse_enet_coordinate_descent(


,model,train_mae,val_mae,val_rmse,val_r2
2,xgb_reg,3.8084,5.0666,7.3167,0.4423
1,rf_reg,3.1905,5.8780,7.8629,0.3559
0,elastic_net,5.3086,6.9945,8.2816,0.2855


In [16]:
best_name = val_table.iloc[0]["model"]
best_pipe = fitted[best_name]

pred_val = best_pipe.predict(X_val)
pred_test = best_pipe.predict(X_test)

test_metrics = {
    "best_model": best_name,
    "test_mae": mean_absolute_error(y_test, pred_test),
    "test_rmse": np.sqrt(mean_squared_error(y_test, pred_test)),
    "test_r2": r2_score(y_test, pred_test),
}

pd.DataFrame([test_metrics]).round(4)

,best_model,test_mae,test_rmse,test_r2
0,xgb_reg,5.1173,7.3109,0.2304


In [17]:
# Convert regression output into a late-risk score: P(margin <= 0)
# We calibrate from validation predictions to keep probability semantics.
val_is_late = (y_val <= 0).astype(int)

def build_risk_mapper(pred_margin_val, y_late_val):
    order = np.argsort(pred_margin_val)
    x = pred_margin_val[order]
    y = y_late_val.values[order]

    # Monotonic: lower predicted margin => higher late risk
    iso = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=False, out_of_bounds="clip")
    iso.fit(x, y)
    return iso

risk_mapper = build_risk_mapper(pred_val, val_is_late)
risk_test = risk_mapper.predict(pred_test)

policy_df = pd.DataFrame({
    "pred_margin": pred_test,
    "risk_late": risk_test,
    "is_late": (y_test <= 0).astype(int).values,
})

# Capacity-constrained thresholding (max alerts = 20% of orders)
max_alert_rate = 0.20
thr_grid = np.linspace(0.05, 0.95, 91)
rows = []
for thr in thr_grid:
    pred_alert = (policy_df["risk_late"] >= thr).astype(int)
    alert_rate = pred_alert.mean()
    if alert_rate > max_alert_rate:
        continue

    tp = int(((pred_alert == 1) & (policy_df["is_late"] == 1)).sum())
    fp = int(((pred_alert == 1) & (policy_df["is_late"] == 0)).sum())
    fn = int(((pred_alert == 0) & (policy_df["is_late"] == 1)).sum())

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    cost = fp * 15 + fn * 155

    rows.append({
        "threshold": thr,
        "alert_rate": alert_rate,
        "precision": prec,
        "recall": rec,
        "business_cost": cost,
    })

policy_table = pd.DataFrame(rows).sort_values("business_cost")
policy_table.head(10).round(4)

,threshold,alert_rate,precision,recall,business_cost
0,0.18,0.1778,0.1841,0.3486,168090
1,0.19,0.1299,0.1750,0.2422,182220
2,0.20,0.0738,0.1596,0.1256,196945
3,0.21,0.0673,0.1617,0.1160,197745
5,0.23,0.0313,0.1552,0.0517,204735
4,0.22,0.0316,0.1535,0.0517,204810
15,0.33,0.0023,0.1818,0.0044,209345
14,0.32,0.0023,0.1818,0.0044,209345
13,0.31,0.0023,0.1818,0.0044,209345
12,0.30,0.0023,0.1818,0.0044,209345


In [19]:
best_policy = policy_table.iloc[0].to_dict()

print("Best policy under 20% alert cap")
for k, v in best_policy.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# Quantile-based risk bands keep segments populated for ops triage.
band_df = policy_df.copy()
band_labels = ["Low", "Medium", "High", "Critical"]
band_df["risk_band"] = pd.qcut(
    band_df["risk_late"],
    q=4,
    labels=band_labels,
    duplicates="drop",
)

band_summary = (
    band_df.groupby("risk_band", observed=False)
    .agg(
        orders=("is_late", "size"),
        late_rate=("is_late", "mean"),
        avg_pred_margin=("pred_margin", "mean"),
    )
    .reset_index()
)
band_summary["late_rate"] = (band_summary["late_rate"] * 100).round(2)
band_summary.round(3)

Best policy under 20% alert cap
threshold: 0.1800
alert_rate: 0.1778
precision: 0.1841
recall: 0.3486
business_cost: 168090.0000


,risk_band,orders,late_rate,avg_pred_margin
0,Low,3829,1.07,16.733999
1,Medium,3857,4.15,10.320000
2,High,3629,15.51,6.471000
3,Critical,3107,18.99,-0.271000


## Upgraded Regression + Risk Pipeline (Transferred)

This section ports the improved regression pipeline from `scripts/model_improvement_v1.py`:
- logistics-aware feature engineering,
- train-only route/seller reliability encodings,
- HGB/XGBoost regression comparison,
- isotonic risk mapping and capacity-capped policy optimization.

In [20]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, precision_score, recall_score, f1_score
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB_REG = True
except Exception:
    HAS_XGB_REG = False

DATA_PATH3 = Path("../datasets/processed/olist_merged_clean.csv")
df3 = pd.read_csv(DATA_PATH3)

feat3 = df3.copy()
for _c in ["order_purchase_timestamp", "order_approved_at", "order_estimated_delivery_date"]:
    feat3[_c] = pd.to_datetime(feat3[_c], errors="coerce")

feat3["purchase_month"] = feat3["order_purchase_timestamp"].dt.month
feat3["purchase_dow"] = feat3["order_purchase_timestamp"].dt.dayofweek
feat3["purchase_hour"] = feat3["order_purchase_timestamp"].dt.hour
feat3["is_weekend"] = feat3["purchase_dow"].isin([5, 6]).astype(int)
feat3["is_peak_season"] = feat3["purchase_month"].isin([11, 12]).astype(int)
feat3["product_volume_cm3"] = feat3["max_length_cm"] * feat3["max_height_cm"] * feat3["max_width_cm"]
feat3["is_bulky"] = ((feat3["max_weight_g"].fillna(0) > 10000) | (feat3["product_volume_cm3"].fillna(0) > 100000)).astype(int)
feat3["freight_ratio"] = feat3["total_freight"] / (feat3["total_price"].abs() + 1)
feat3["weight_to_price"] = feat3["max_weight_g"] / (feat3["total_price"].abs() + 1)
feat3["distance_x_weight"] = feat3["distance_km"] * feat3["max_weight_g"]
feat3["route_key"] = feat3["seller_state"].astype(str) + "->" + feat3["customer_state"].astype(str)
feat3["delivery_margin_days"] = feat3["estimated_delivery_days"] - feat3["actual_delivery_days"]

feat3 = feat3.dropna(subset=["order_purchase_timestamp", "actual_delivery_days", "estimated_delivery_days", "delivery_margin_days"])
feat3 = feat3[(feat3["actual_delivery_days"] >= 0) & (feat3["actual_delivery_days"] <= 120)]
feat3 = feat3[(feat3["estimated_delivery_days"] >= 0) & (feat3["estimated_delivery_days"] <= 120)]
feat3 = feat3[(feat3["delivery_margin_days"] >= -120) & (feat3["delivery_margin_days"] <= 120)]

t_train3 = feat3["order_purchase_timestamp"].quantile(0.70)
t_val3 = feat3["order_purchase_timestamp"].quantile(0.85)
feat3["split3"] = np.where(
    feat3["order_purchase_timestamp"] <= t_train3,
    "train",
    np.where(feat3["order_purchase_timestamp"] <= t_val3, "val", "test"),
)

tr3 = feat3[feat3["split3"] == "train"].copy()
va3 = feat3[feat3["split3"] == "val"].copy()
te3 = feat3[feat3["split3"] == "test"].copy()

global_late3 = tr3["is_late"].mean()
route_stats3 = tr3.groupby("route_key")["is_late"].agg(["mean", "count"]).reset_index()
route_stats3["route_late_rate_te"] = (route_stats3["mean"] * route_stats3["count"] + global_late3 * 20) / (route_stats3["count"] + 20)
route_map3 = dict(zip(route_stats3["route_key"], route_stats3["route_late_rate_te"]))

seller_stats3 = tr3.groupby("seller_zip_code_prefix")["is_late"].agg(["mean", "count"]).reset_index()
seller_stats3["seller_late_rate_te"] = (seller_stats3["mean"] * seller_stats3["count"] + global_late3 * 30) / (seller_stats3["count"] + 30)
seller_map3 = dict(zip(seller_stats3["seller_zip_code_prefix"], seller_stats3["seller_late_rate_te"]))

for _fr in (tr3, va3, te3):
    _fr["route_late_rate_te"] = _fr["route_key"].map(route_map3).fillna(global_late3)
    _fr["seller_late_rate_te"] = _fr["seller_zip_code_prefix"].map(seller_map3).fillna(global_late3)

num_f3 = [
    "distance_km", "total_price", "total_freight", "max_weight_g", "n_items",
    "payment_installments", "estimated_delivery_days", "product_volume_cm3", "freight_ratio",
    "weight_to_price", "distance_x_weight", "route_late_rate_te", "seller_late_rate_te",
    "purchase_month", "purchase_dow", "purchase_hour", "is_weekend", "is_peak_season",
    "is_bulky", "same_state"
]
cat_f3 = ["payment_type", "customer_state", "seller_state"]
all_f3 = num_f3 + cat_f3

Xtr3, ytr3 = tr3[all_f3], tr3["delivery_margin_days"]
Xva3, yva3 = va3[all_f3], va3["delivery_margin_days"]
Xte3, yte3 = te3[all_f3], te3["delivery_margin_days"]

pre3 = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler(with_mean=False))]), num_f3),
        ("cat", Pipeline(steps=[("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01))]), cat_f3),
    ],
    remainder="drop",
)

models3 = {
    "hgb_reg": Pipeline(
        steps=[
            ("pre", pre3),
            ("reg", HistGradientBoostingRegressor(learning_rate=0.06, max_depth=7, max_iter=450, random_state=42)),
        ]
    )
}
if HAS_XGB_REG:
    models3["xgb_reg"] = Pipeline(
        steps=[
            ("pre", pre3),
            ("reg", XGBRegressor(
                n_estimators=550, learning_rate=0.05, max_depth=7,
                subsample=0.9, colsample_bytree=0.85,
                objective="reg:squarederror", random_state=42, n_jobs=-1,
            )),
        ]
    )

rows3, fitted3 = [], {}
for _name, _m in models3.items():
    _m.fit(Xtr3, ytr3)
    _p = _m.predict(Xva3)
    rows3.append({
        "model": _name,
        "val_mae": mean_absolute_error(yva3, _p),
        "val_rmse": np.sqrt(mean_squared_error(yva3, _p)),
        "val_r2": r2_score(yva3, _p),
    })
    fitted3[_name] = _m

improved_reg_val_summary = pd.DataFrame(rows3).sort_values("val_mae")
best_reg_name3 = improved_reg_val_summary.iloc[0]["model"]
best_reg_model3 = fitted3[best_reg_name3]

pred_val3 = best_reg_model3.predict(Xva3)
pred_test3 = best_reg_model3.predict(Xte3)

improved_reg_test_metrics = pd.DataFrame([
    {
        "best_model": best_reg_name3,
        "test_mae": mean_absolute_error(yte3, pred_test3),
        "test_rmse": np.sqrt(mean_squared_error(yte3, pred_test3)),
        "test_r2": r2_score(yte3, pred_test3),
    }
])

val_late3 = (yva3 <= 0).astype(int)
risk_iso3 = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=False, out_of_bounds="clip")
risk_iso3.fit(pred_val3, val_late3)
risk_test3 = risk_iso3.predict(pred_test3)

test_late3 = (yte3 <= 0).astype(int)
best_pol3 = None
for _t in np.linspace(0.05, 0.95, 91):
    _pred = (risk_test3 >= _t).astype(int)
    _alert = _pred.mean()
    if _alert > 0.20:
        continue
    _tp = int(((_pred == 1) & (test_late3.values == 1)).sum())
    _fp = int(((_pred == 1) & (test_late3.values == 0)).sum())
    _fn = int(((_pred == 0) & (test_late3.values == 1)).sum())
    _row = {
        "best_model": best_reg_name3,
        "threshold": float(_t),
        "alert_rate": float(_alert),
        "precision": precision_score(test_late3, _pred, zero_division=0),
        "recall": recall_score(test_late3, _pred, zero_division=0),
        "f1": f1_score(test_late3, _pred, zero_division=0),
        "business_cost": int(_fp * 15 + _fn * 155),
    }
    if (best_pol3 is None) or (_row["business_cost"] < best_pol3["business_cost"]):
        best_pol3 = _row

improved_reg_risk_policy = pd.DataFrame([best_pol3])

display(improved_reg_val_summary.round(4))
display(improved_reg_test_metrics.round(4))
display(improved_reg_risk_policy.round(4))

,model,val_mae,val_rmse,val_r2
0,hgb_reg,4.7203,6.4599,0.5651
1,xgb_reg,4.8482,6.6160,0.5438


,best_model,test_mae,test_rmse,test_r2
0,hgb_reg,4.1267,5.6218,0.5452


,best_model,threshold,alert_rate,precision,recall,f1,business_cost
0,hgb_reg,0.15,0.0535,0.1684,0.1444,0.1555,128980
